# Qwen 1.5B Colab benchmark bridge

This notebook is the minimal upload-and-run bridge for Colab. It fixes the repo import path, loads the local project code, and runs the benchmark against Qwen 1.5B without needing any local model load.


In [ ]:
# Cell 1: repo import path + project import
!python -V
!pip install -q --upgrade pip
!pip install -q transformers accelerate sentencepiece

import os
import sys
from pathlib import Path

repo_candidates = [
    Path('/content/kv-eviction'),
    Path('/workspace/kv-eviction'),
    Path.cwd() / 'kv-eviction',
    Path.cwd(),
]

repo_root = None
for candidate in repo_candidates:
    if candidate.exists() and (candidate / 'src').exists():
        repo_root = candidate
        break

if repo_root is None:
    possible = sorted(Path.cwd().glob('**/src'), key=lambda p: len(p.parts))
    if possible:
        repo_root = possible[0].parent

if repo_root is None:
    raise FileNotFoundError('Repo not found. Clone or mount it to /content/kv-eviction or /workspace/kv-eviction.')

sys.path.insert(0, str(repo_root))
print('repo_root =', repo_root)

from src.eviction import EvictionManager, WindowState
from src.index_map import IndexMap
from src.rope import apply_rope, precompute_rope_freqs
from src.shadow_cache import ShadowCache

print('imports_ok = True')


In [ ]:
# Cell 2: sanity-check the package after import
import torch

freqs = precompute_rope_freqs(1024, 8, device=torch.device('cpu'))
raw_key = torch.randn(8, dtype=torch.float32)
shadow_cache = ShadowCache(survivor_every=8)
shadow_cache.add(token_id=7, raw_key=raw_key, rotated_key=raw_key.clone(), is_survivor=True)
index_map = IndexMap()
index_map.compact([7])
manager = EvictionManager(freqs)
boundary = manager.on_boundary(shadow_cache, index_map, WindowState(window_size=64, evict_n=8))
rotated = boundary['rotated'][0]['key']
expected = apply_rope(
    raw_key.unsqueeze(0).unsqueeze(0),
    torch.tensor([0.0], dtype=torch.float32),
    freqs,
).squeeze(0).squeeze(0)
print('smoke_max_abs_diff =', (rotated - expected).abs().max().item())
assert (rotated - expected).abs().max().item() < 1e-5


In [ ]:
# Cell 3: run the real Qwen 1.5B benchmark
import re
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
secret_text = 'The exact secret is BAKED-42.'

print('loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('loading model ...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()


def make_prompt(secret_index, sentence_count=8):
    sentences = []
    for i in range(sentence_count):
        if i == secret_index:
            sentences.append(f'Sentence {i}: {secret_text}')
        else:
            sentences.append(f'Sentence {i}: the answer is not the secret fact.')
    return (
        'You are given a short list of sentences. Return only the exact sentence that contains the secret phrase.\n'
        + '\n'.join(sentences)
        + '\nQuestion: Which sentence contains the secret phrase? Reply with only that sentence.'
    )


def norm(s):
    return re.sub(r'\s+', ' ', s).strip().lower()

results = []
for secret_index in [0, 1, 2, 4, 7]:
    prompt = make_prompt(secret_index)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs.get('attention_mask'),
            max_new_tokens=24,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    exact = 1.0 if norm(secret_text) in norm(generated) or norm(generated) in norm(secret_text) else 0.0
    fuzzy = 1.0 if norm(secret_text.split()[-1]) in norm(generated) else 0.0
    row = {
        'secret_index': secret_index,
        'exact_match': exact,
        'fuzzy_match': fuzzy,
        'generated_text': generated,
    }
    results.append(row)
    print(json.dumps(row, sort_keys=True))

print('done')


## Notes

- This is the exact notebook to upload to Colab.
- Run the cells in order.
- The laptop should not load the model; only the Colab T4 runtime should.
